## Setup

In [19]:
import os
import math
import pandas as pd
from pathlib import Path
from csv import QUOTE_NONE

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType
from pyspark.ml.recommendation import ALSModel

# Load

In [20]:
LOCAL_DATA_ROOT = Path(os.environ.get("MIND_DATA_ROOT", "../smallDataset")).expanduser().resolve()

train_dir = LOCAL_DATA_ROOT / f"MINDsmall_train"
valid_dir = LOCAL_DATA_ROOT / f"MINDsmall_dev"

train_news_path      = str(train_dir / "news.tsv")
train_behaviors_path = str(train_dir / "behaviors.tsv")
valid_news_path      = str(valid_dir / "news.tsv")
valid_behaviors_path = str(valid_dir / "behaviors.tsv")

NEWS_COLUMNS = [
    "nid",
    "vertical",
    "subvertical",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

def load_tsv(path_str: str, columns):
    return pd.read_table(
        path_str,
        header=None,
        names=columns,
        sep='\t',
        quoting=QUOTE_NONE,
        dtype=object,
        na_filter=False,
    )

train_news_df = load_tsv(train_news_path, NEWS_COLUMNS)
valid_news_df = load_tsv(valid_news_path, NEWS_COLUMNS)
train_behaviors_df = load_tsv(train_behaviors_path, BEHAVIOR_COLUMNS)
valid_behaviors_df = load_tsv(valid_behaviors_path, BEHAVIOR_COLUMNS)

print(f"Loaded {len(train_news_df)} train news rows and {len(valid_news_df)} validation news rows from {LOCAL_DATA_ROOT}")
print(f"Loaded {len(train_behaviors_df)} train behaviors rows and {len(valid_behaviors_df)} validation behaviors rows")

Loaded 51282 train news rows and 42416 validation news rows from /home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset
Loaded 156965 train behaviors rows and 73152 validation behaviors rows


In [21]:
spark = (
    SparkSession.builder
    .appName('mind-als-demo')
    .getOrCreate()
)


## Load models

In [22]:
als_model = ALSModel.load("../models/als_model")


## Define maturity aware model

In [23]:
from collections import Counter
from typing import List, Tuple

MATURITY_THRESHOLD = 5


def parse_impression_tokens(raw_impressions: str) -> List[Tuple[str, int]]:
    tokens: List[Tuple[str, int]] = []
    if not isinstance(raw_impressions, str):
        return tokens
    for token in raw_impressions.split():
        if '-' not in token:
            continue
        nid, label = token.rsplit('-', 1)
        if not nid:
            continue
        try:
            clicked = int(label)
        except ValueError:
            continue
        tokens.append((nid, 1 if clicked > 0 else 0))
    return tokens


def build_interactions(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in df.itertuples(index=False):
        parsed = parse_impression_tokens(row.impressions)
        if not parsed:
            continue
        for nid, clicked in parsed:
            rows.append((row.user_id, nid, float(clicked)))
    return pd.DataFrame(rows, columns=['user_id', 'nid', 'label'])


def build_popularity_stats(behaviors_df: pd.DataFrame) -> pd.DataFrame:
    click_counter = Counter()
    display_counter = Counter()
    for impression_str in behaviors_df['impressions']:
        for nid, clicked in parse_impression_tokens(impression_str):
            display_counter[nid] += 1
            if clicked:
                click_counter[nid] += 1
    if not display_counter:
        return pd.DataFrame(columns=['nid', 'impressions', 'clicks', 'ctr', 'score'])
    stats = pd.DataFrame(
        {
            'nid': list(display_counter.keys()),
            'impressions': list(display_counter.values()),
        }
    )
    stats['clicks'] = stats['nid'].map(lambda nid: click_counter.get(nid, 0))
    stats['ctr'] = stats['clicks'] / stats['impressions'].replace(0, pd.NA)
    stats['ctr'] = stats['ctr'].fillna(0.0)
    stats['score'] = stats['clicks'] + stats['ctr']
    stats = stats.sort_values(by=['score', 'impressions'], ascending=False).reset_index(drop=True)
    return stats


train_interactions_pd = build_interactions(train_behaviors_df)
valid_interactions_pd = build_interactions(valid_behaviors_df)
print(f"Prepared {len(train_interactions_pd):,} train interactions and {len(valid_interactions_pd):,} validation interactions")

train_spark_df = spark.createDataFrame(train_interactions_pd)
valid_spark_df = spark.createDataFrame(valid_interactions_pd)

user_indexer = StringIndexer(inputCol='user_id', outputCol='user_idx', handleInvalid='skip').fit(train_spark_df)
item_indexer = StringIndexer(inputCol='nid', outputCol='item_idx', handleInvalid='skip').fit(train_spark_df)

train_indexed_df = item_indexer.transform(user_indexer.transform(train_spark_df))
valid_indexed_df = item_indexer.transform(user_indexer.transform(valid_spark_df))

train_ratings_df = train_indexed_df.select(
    F.col('user_idx').cast('int').alias('user_idx'),
    F.col('item_idx').cast('int').alias('item_idx'),
    F.col('label').cast('float').alias('label'),
).cache()

valid_ratings_df = valid_indexed_df.select(
    F.col('user_idx').cast('int').alias('user_idx'),
    F.col('item_idx').cast('int').alias('item_idx'),
    F.col('label').cast('float').alias('label'),
    'nid',
).cache()

print(f"Spark train interactions: {train_ratings_df.count():,}")
print(f"Spark validation interactions: {valid_ratings_df.count():,}")

popularity_stats = build_popularity_stats(train_behaviors_df)
popularity_rows = [(row.nid, float(row.score)) for row in popularity_stats.itertuples(index=False)]
if popularity_rows:
    popularity_scores_spark_df = spark.createDataFrame(popularity_rows, ['nid', 'popularity_score'])
else:
    popularity_scores_spark_df = spark.createDataFrame([], 'nid string, popularity_score double')
fallback_popularity_score = float(popularity_stats['score'].min()) - 1.0 if not popularity_stats.empty else -1.0

if not popularity_stats.empty:
    top_popular = popularity_stats.loc[:9, ['nid', 'clicks', 'impressions', 'ctr']].copy()
    top_popular['ctr'] = top_popular['ctr'].round(4)
    print("Top globally popular news items:")
    print(top_popular.to_string(index=False))

user_activity_df = (
    train_ratings_df
    .groupBy('user_idx')
    .agg(F.count('*').alias('train_impressions'))
)

als_predictions_df = als_model.transform(valid_ratings_df).select('user_idx', 'item_idx', 'nid', 'label', 'prediction')

maturity_features_df = (
    als_predictions_df
    .join(popularity_scores_spark_df, 'nid', 'left')
    .withColumn('popularity_score', F.coalesce(F.col('popularity_score'), F.lit(fallback_popularity_score)))
    .join(user_activity_df, 'user_idx', 'left')
    .withColumn('train_impressions', F.coalesce(F.col('train_impressions'), F.lit(0)))
    .withColumnRenamed('prediction', 'cf_prediction')
)

maturity_condition = F.col('train_impressions') <= F.lit(MATURITY_THRESHOLD)

predictions = (
    maturity_features_df
    .withColumn(
        'prediction',
        F.when(maturity_condition, F.col('popularity_score')).otherwise(F.col('cf_prediction')),
    )
    .select('user_idx', 'item_idx', 'label', 'prediction')
    .cache()
)

routing_summary = maturity_features_df.agg(
    F.sum(F.when(maturity_condition, 1).otherwise(0)).alias('popularity_rows'),
    F.sum(F.when(F.col('train_impressions') > F.lit(MATURITY_THRESHOLD), 1).otherwise(0)).alias('cf_rows'),
).first()
pop_rows = int((routing_summary['popularity_rows'] or 0)) if routing_summary else 0
cf_rows = int((routing_summary['cf_rows'] or 0)) if routing_summary else 0
print(
    f"Impression routing → popularity: {pop_rows:,}, "
    f"ALS: {cf_rows:,}, total: {pop_rows + cf_rows:,}"
)


Prepared 5,843,444 train interactions and 2,740,998 validation interactions


26/03/20 10:12:14 WARN TaskSetManager: Stage 9 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:21 WARN TaskSetManager: Stage 15 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:26 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:26 WARN TaskSetManager: Stage 21 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:31 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:31 WARN TaskSetManager: Stage 22 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:32 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


Spark train interactions: 5,843,444


26/03/20 10:12:32 WARN TaskSetManager: Stage 25 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:34 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:34 WARN TaskSetManager: Stage 26 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.


Spark validation interactions: 265,183
Top globally popular news items:
   nid  clicks  impressions    ctr
N55689    4316        18315 0.2357
N35729    3346        15418 0.2170
N33619    3246        15062 0.2155
N53585    2835         9908 0.2861
N63970    2578        14276 0.1806
N49685    2294         7229 0.3173
N49279    2270         6229 0.3644
  N287    2128        10019 0.2124
N23446    1930        15500 0.1245
N51048    1875        19242 0.0974


26/03/20 10:12:38 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:39 WARN TaskSetManager: Stage 32 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:40 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/20 10:12:40 WARN TaskSetManager: Stage 37 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.


Impression routing → popularity: 5,195, ALS: 259,988, total: 265,183


## Ranking metrics

In [24]:
LOG_2 = math.log(2.0)
rank_window = Window.partitionBy('user_idx').orderBy(F.desc('prediction'))
ranked_predictions_df = predictions.withColumn('rank', F.row_number().over(rank_window)).cache()

user_label_stats_df = valid_ratings_df.groupBy('user_idx').agg(
    F.sum('label').alias('num_pos'),
    F.count('*').alias('num_interactions'),
)


def make_ideal_dcg_udf(k: int):
    def ideal(count):
        if count is None:
            return 0.0
        upto = min(int(count), k)
        score = 0.0
        for i in range(upto):
            score += 1.0 / math.log2(i + 2)
        return float(score)

    return F.udf(ideal, DoubleType())


ideal_dcg_5_udf = make_ideal_dcg_udf(5)
ideal_dcg_10_udf = make_ideal_dcg_udf(10)

user_label_stats_df = (
    user_label_stats_df
    .withColumn('ideal_dcg_5', ideal_dcg_5_udf('num_pos'))
    .withColumn('ideal_dcg_10', ideal_dcg_10_udf('num_pos'))
)


def summarize_topk(topk_df, k: int):
    """Aggregate DCG and hit contributions for cutoff k."""
    contrib_df = topk_df.withColumn(
        'dcg_contrib',
        F.col('label') * (F.lit(LOG_2) / F.log(F.col('rank') + F.lit(1.0))),
    )
    return contrib_df.groupBy('user_idx').agg(
        F.sum('dcg_contrib').alias(f'dcg_{k}'),
        F.max('label').alias(f'hit_{k}'),
    )


top5_metrics_df = summarize_topk(ranked_predictions_df.filter(F.col('rank') <= 5), 5)
top10_metrics_df = summarize_topk(ranked_predictions_df.filter(F.col('rank') <= 10), 10)

metrics_per_user_df = (
    user_label_stats_df
    .join(top5_metrics_df, 'user_idx', 'left')
    .join(top10_metrics_df, 'user_idx', 'left')
    .fillna({'dcg_5': 0.0, 'dcg_10': 0.0, 'hit_5': 0.0, 'hit_10': 0.0})
)

metrics_per_user_df = (
    metrics_per_user_df
    .withColumn('ndcg_5', F.when(F.col('ideal_dcg_5') > 0, F.col('dcg_5') / F.col('ideal_dcg_5')))
    .withColumn('ndcg_10', F.when(F.col('ideal_dcg_10') > 0, F.col('dcg_10') / F.col('ideal_dcg_10')))
)

ranking_summary_row = metrics_per_user_df.agg(
    F.mean('ndcg_5').alias('ndcg_5'),
    F.mean('ndcg_10').alias('ndcg_10'),
    F.mean('hit_5').alias('hit_5'),
    F.mean('hit_10').alias('hit_10'),
).first()
ranking_summary = ranking_summary_row.asDict() if ranking_summary_row else {}

first_relevant_df = (
    ranked_predictions_df
    .filter(F.col('label') > 0)
    .groupBy('user_idx')
    .agg(F.min('rank').alias('first_rank'))
)

rr_df = metrics_per_user_df.select('user_idx').join(first_relevant_df, 'user_idx', 'left').withColumn(
    'rr',
    F.when(F.col('first_rank').isNotNull(), 1.0 / F.col('first_rank')).otherwise(0.0),
)
mrr_row = rr_df.agg(F.mean('rr').alias('mrr')).first()
mrr_value = mrr_row['mrr'] if mrr_row else None

score_and_labels_rdd = predictions.select('prediction', 'label').rdd.map(
    lambda row: (float(row.prediction), float(row.label))
)
auc_value = BinaryClassificationMetrics(score_and_labels_rdd).areaUnderROC


def to_float(value):
    return 0.0 if value is None else float(value)

ndcg5 = to_float(ranking_summary.get('ndcg_5'))
ndcg10 = to_float(ranking_summary.get('ndcg_10'))
hit5 = to_float(ranking_summary.get('hit_5'))
hit10 = to_float(ranking_summary.get('hit_10'))
mrr = to_float(mrr_value)
auc = to_float(auc_value)

print(f"nDCG@5: {ndcg5:.4f}")
print(f"nDCG@10: {ndcg10:.4f}")
print(f"Hit@5: {hit5:.4f}")
print(f"Hit@10: {hit10:.4f}")
print(f"MRR: {mrr:.4f}")
print(f"AUC: {auc:.4f}")

ranked_predictions_df.unpersist()

26/03/20 10:12:42 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:42 WARN TaskSetManager: Stage 43 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:43 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/20 10:12:43 WARN TaskSetManager: Stage 46 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:45 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:46 WARN TaskSetManager: Stage 54 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.
26/03/20 10:12:50 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/20 10:12:50 WARN TaskSetManager: Stage 62 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.
/home/asmvas/.venvs/recommenders/lib/python3.11/site-packages/pyspark/sql/context.py:15

nDCG@5: 0.2778
nDCG@10: 0.3346
Hit@5: 0.4394
Hit@10: 0.5935
MRR: 0.2754
AUC: 0.5998


DataFrame[user_idx: int, item_idx: int, label: float, prediction: double, rank: int]